In [1]:
!pip install eyepop==3.12.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: cryptography
    Found existing installation: cryptography 49.0.0
    Uninstalling cryptography-49.0.0:
      Successfully uninstalled cryptography-49.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyopenssl 26.3.0 requires cryptography<50,>=49.0.0, but you have cryptography 46.0.7 which is incompatible.


In [2]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

Enter your Account UUID: a5184defa8e847248f589d35080efbfa
Enter your API KEY: ··········


In [3]:
NAMESPACE_PREFIX="datasciencealliance-org" # Add your namespace-prefix here

### Define Ability

In [4]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import InferenceComponent, Pop
import json


DRIVE_THRU_QUEUE_HEALTH_PROMPT = """
Analyze the quick service restaurant drive-thru image and return structured queue health information.

Return only valid JSON. Do not include markdown, explanations, or extra commentary.

Use this exact JSON structure:

{
  "cars_waiting": null,
  "car_count": null,
  "estimated_wait_time": null,
  "queue_extending_beyond_expected_area": null,
  "overflow_location": null,
  "bottleneck_location": null,
  "peak_traffic_period": null,
  "visual_evidence": null
}

Instructions:

- "cars_waiting" should be true if one or more cars are visibly waiting in the drive-thru lane or drive-thru queue. Otherwise, use false.

- "car_count" should estimate the number of visible cars waiting in the drive-thru queue. Count only cars that appear to be part of the drive-thru queue. Do not count parked cars unless they are clearly waiting in the drive-thru line.

- "estimated_wait_time" should be one of: "none", "low", "medium", "high", "very_high", or "unknown".
Use "none" when there are no cars waiting.
Use "low" for about 1 to 3 cars waiting.
Use "medium" for about 4 to 7 cars waiting.
Use "high" for about 8 to 12 cars waiting.
Use "very_high" for more than 12 cars waiting, queue overflow, or severe congestion.
Use "unknown" if the image does not show enough of the drive-thru lane to estimate.

- "queue_extending_beyond_expected_area" should be true if the drive-thru queue extends outside the normal drive-thru lane, spills into the parking lot aisle, blocks parked cars, reaches the driveway entrance, or extends toward the street. Otherwise, use false.

- "overflow_location" should describe where the queue extends beyond the expected drive-thru area. Use values such as "none", "parking_lot_aisle", "driveway_entrance", "street", "blocked_parking_spaces", "unknown", or a short phrase if needed.

- "bottleneck_location" should identify the main visible bottleneck. Use one of: "none", "order_speaker", "payment_window", "pickup_window", "drive_thru_entrance", "merge_point", "overall_queue", "unknown".
Use "order_speaker" when cars are backed up before or at the menu/order point while the area after the speaker is clearer.
Use "payment_window" when cars are backed up around the payment window.
Use "pickup_window" when cars are stacked near the final pickup window.
Use "drive_thru_entrance" or "merge_point" when cars are congested before entering the official drive-thru lane.
Use "overall_queue" when the entire drive-thru lane is long or congested without one clear bottleneck.

- "peak_traffic_period" should estimate the apparent traffic period from the visible scene. Use one of: "off_peak", "breakfast_rush", "lunch_rush", "dinner_rush", "peak_period_unknown", or "unknown".
Use "off_peak" when only a few cars are present and the lot is calm.
Use "breakfast_rush" if the image shows morning lighting and commuter-style traffic.
Use "lunch_rush" if the image shows bright midday lighting with many cars.
Use "dinner_rush" if the image shows evening lighting, headlights, or a dinner-time rush.
Use "peak_period_unknown" when the queue is clearly heavy but the time of day is unclear.
Use "unknown" if there is not enough visual evidence.

- "visual_evidence" should briefly describe the visible cues supporting the assessment, such as number of cars, where the queue is located, whether it overflows, and where cars appear backed up.

Important visual rules:
- Focus only on the drive-thru lane, queue, restaurant exterior, parking lot aisle, driveway entrance, and street access.
- Do not count unrelated parked cars as waiting unless they are visibly in the drive-thru queue.
- Do not rely on readable signs, timestamps, text overlays, or map graphics.
- Ignore brand logos, restaurant names, captions, UI overlays, timestamps, and watermarks.
- If the image is ambiguous, use "unknown" for uncertain fields instead of guessing.

Return only the JSON object.
"""


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.describe.drive-thru-queue-health",
        description="Analyze quick service restaurant drive-thru camera images and return structured queue health details, including cars waiting, estimated wait time, queue overflow, bottleneck location, and apparent peak traffic period",
        worker_release="qwen3-instruct",
        text_prompt=DRIVE_THRU_QUEUE_HEALTH_PROMPT,
        transform_into=TransformInto(),
        config=InferRuntimeConfig(
            max_new_tokens=300,
            image_size=640
        ),
        is_public=False
    )
]

### Create Ability

In [5]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

created ability 06a612410a15757e80008d427dd3c6ae with alias entries [AbilityAliasEntry(alias='datasciencealliance-org.describe.drive-thru-queue-health', tag='1.0.1'), AbilityAliasEntry(alias='datasciencealliance-org.describe.drive-thru-queue-health', tag='latest')]


### Evalulate on a Single Image

In [6]:
from pathlib import Path
import json
from eyepop import EyePopSdk
from eyepop.worker.worker_types import InferenceComponent, Pop


def extract_json_from_result(result):
    """
    Extract structured JSON from EyePop describe output.
    """
    for text_item in result.get("texts", []):
        text = text_item.get("text", "").strip()

        try:
            return json.loads(text)
        except json.JSONDecodeError:
            # Try to recover if the model wrapped JSON in extra text.
            start = text.find("{")
            end = text.rfind("}")

            if start != -1 and end != -1 and end > start:
                json_text = text[start:end + 1]

                try:
                    return json.loads(json_text)
                except json.JSONDecodeError:
                    pass

    return None


pop = Pop(components=[
    InferenceComponent(
        ability=f"{NAMESPACE_PREFIX}.describe.drive-thru-queue-health:latest"
    )
])


# Replace this with your actual image path.
input_path = Path("/content/sample_drive_thru_image.jpg")


raw_results = []

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
    endpoint.set_pop(pop)

    job = endpoint.upload(input_path)

    while result := job.predict():
        raw_results.append(result)


print("=== RAW EYEPOP RESULT ===")
print(json.dumps(raw_results[:1], indent=2))


print("\n=== DRIVE-THRU QUEUE HEALTH OUTPUT ===")

if raw_results:
    structured_output = extract_json_from_result(raw_results[0])

    if structured_output:
        print(json.dumps(structured_output, indent=2))
    else:
        print("No valid JSON detected.")
else:
    print("No result returned.")

=== RAW EYEPOP RESULT ===
[
  {
    "compute_units": 0.327,
    "seconds": 0,
    "source_height": 559,
    "source_id": "9ab8cf96-8609-11f1-9495-52da53fa7a4c",
    "source_width": 1024,
    "system_timestamp": 1784751129817254000,
    "texts": [
      {
        "id": 1,
        "text": "```json\n{\n  \"cars_waiting\": true,\n  \"car_count\": 14,\n  \"estimated_wait_time\": \"very_high\",\n  \"queue_extending_beyond_expected_area\": true,\n  \"overflow_location\": \"parking_lot_aisle\",\n  \"bottleneck_location\": \"overall_queue\",\n  \"peak_traffic_period\": \"dinner_rush\",\n  \"visual_evidence\": \"A long line of cars is backed up in the drive-thru lane, extending significantly past the building and spilling into the parking lot aisle. The queue includes at least 14 vehicles, with cars waiting in the main lane and others visible in the overflow area. The lighting suggests evening, consistent with a dinner rush.\"\n}\n```"
      }
    ],
    "timestamp": 0
  }
]

=== DRIVE-THRU QUEU